> **Note**: This notebook has been upgraded to reflect the Phase 12 Final ML Pipeline Upgrade, utilizing the expanded 18-class taxonomy, ONNX export workflows, and accurate Boolean-mask object density calculation.

# 📊 PlasticSense AI — Model Evaluation (Notebook 07)

### 🌟 Overview
This notebook performs a **comprehensive, publication-quality evaluation** of the trained YOLOv11 model on the **test dataset**. It is designed to meet **IEEE Final Year Project** standards and produces all metrics, visualizations, and reports required for the research paper.

### 📥 Inputs
| Asset | Path |
|---|---|
| Trained Model | `PlasticSense_AI/models/best.pt` |
| Test Dataset | `PlasticSense_AI/datasets/augmented_yolo/images/test` |

### 📤 Outputs
All results are saved under `PlasticSense_AI/evaluation/` with the following sub-structure:
```
evaluation/
├── metrics/          # JSON & CSV metric reports
├── plots/            # Publication-quality figures (PNG, PDF, SVG)
├── predictions/      # Annotated test image predictions
└── reports/          # Summary evaluation & performance reports
```

### ⚠️ Prerequisites
Run notebooks **01–06** first. This notebook does **NOT** retrain the model.

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 1. Environment Setup & Library Installation
Install and import all required dependencies for evaluation.

In [18]:
!pip install -q ultralytics rich pyyaml pandas opencv-python matplotlib seaborn pillow scikit-learn psutil tqdm

In [19]:
# ──────────────────────────────────────────────────────────
# Standard Library
# ──────────────────────────────────────────────────────────
import os
import sys
import json
import time
import glob
import random
import logging
import datetime
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from collections import defaultdict

# ──────────────────────────────────────────────────────────
# Third-Party
# ──────────────────────────────────────────────────────────
import yaml
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import psutil

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import track

from ultralytics import YOLO

# ──────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.figsize': (12, 8),
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})
sns.set_theme(style='whitegrid', palette='deep')

console = Console()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

console.print('[bold green]✔ All libraries imported successfully.[/bold green]')

✔ All libraries imported successfully.

---
## 2. Logging System Initialization
Set up a dual-output logger (console + file) consistent with prior notebooks.

In [20]:
def setup_logger(log_dir: Path, name: str = 'PlasticSense_Eval') -> logging.Logger:
    """Create a logger with file and console handlers."""
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers = []  # Reset

    fmt = logging.Formatter(
        '[%(asctime)s] %(levelname)s — %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # File handler
    fh = logging.FileHandler(log_dir / 'evaluation.log', mode='w')
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    # Console handler
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    return logger

---
## 3. Hardware Verification
Detect available compute accelerator (CUDA / MPS / CPU).

In [21]:
def check_hardware() -> str:
    """Detect hardware accelerator and display status table."""
    table = Table(title='Hardware & Environment Status', show_header=True)
    table.add_column('Component', style='cyan')
    table.add_column('Status / Version', justify='right')

    table.add_row('Python Version', sys.version.split()[0])
    table.add_row('PyTorch Version', torch.__version__)

    device_type = 'cpu'
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        table.add_row('GPU Accelerator', f'[green]✔ {device_name}[/green]')
        table.add_row('CUDA Version', str(torch.version.cuda))
        # Fix: Change total_mem to total_memory
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        table.add_row('VRAM', f'{vram:.1f} GB')
        device_type = 'cuda:0'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        table.add_row('GPU Accelerator', '[green]✔ Apple Silicon (MPS)[/green]')
        device_type = 'mps'
    else:
        table.add_row('GPU Accelerator', '[bold red]✖ CPU ONLY[/bold red]')

    import ultralytics
    table.add_row('Ultralytics Version', ultralytics.__version__)
    table.add_row('RAM Available', f'{psutil.virtual_memory().available / 1e9:.1f} GB')

    console.print(table)
    return device_type

DEVICE = check_hardware()

      Hardware & Environment Status       
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Component           ┃ Status / Version ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Python Version      │          3.12.13 │
│ PyTorch Version     │     2.11.0+cu128 │
│ GPU Accelerator     │       ✔ Tesla T4 │
│ CUDA Version        │             12.8 │
│ VRAM                │          15.6 GB │
│ Ultralytics Version │          8.4.110 │
│ RAM Available       │          11.5 GB │
└─────────────────────┴──────────────────┘

---
## 4. Project Paths & Directory Structure
Define all paths and create the evaluation output directory tree.

In [24]:
# ──────────────────────────────────────────────────────────
# Detect Environment: Colab vs Local
# ──────────────────────────────────────────────────────────
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
else:
    PROJECT_ROOT = Path('/Users/siddhivinayak/project/PlasticSense AI/Ml-model')

# ──────────────────────────────────────────────────────────
# Input Paths
# ──────────────────────────────────────────────────────────
# Updated: Changed 'Datasets' to lowercase 'datasets' to match file system
DATASET_DIR    = PROJECT_ROOT / 'datasets' / 'augmented_yolo'
DATASET_YAML   = DATASET_DIR / 'dataset.yaml'
MODELS_DIR     = PROJECT_ROOT / 'models'
BEST_PT_PATH   = MODELS_DIR / 'best.pt'
TEST_IMAGES    = DATASET_DIR / 'images' / 'test'
TEST_LABELS    = DATASET_DIR / 'labels' / 'test'

# ──────────────────────────────────────────────────────────
# Output Paths
# ──────────────────────────────────────────────────────────
EVAL_DIR       = PROJECT_ROOT / 'evaluation'
METRICS_DIR    = EVAL_DIR / 'metrics'
PLOTS_DIR      = EVAL_DIR / 'plots'
PREDICTIONS_DIR = EVAL_DIR / 'predictions'
REPORTS_DIR    = EVAL_DIR / 'reports'
LOGS_DIR       = EVAL_DIR / 'logs'

# Create all output directories
for d in [EVAL_DIR, METRICS_DIR, PLOTS_DIR, PREDICTIONS_DIR, REPORTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Initialize logger
logger = setup_logger(LOGS_DIR)
logger.info('Evaluation pipeline initialized.')

console.print(Panel.fit(
    f'[bold cyan]Project Root:[/bold cyan]  {PROJECT_ROOT}\n'
    f'[bold cyan]Model Path:[/bold cyan]    {BEST_PT_PATH}\n'
    f'[bold cyan]Test Images:[/bold cyan]   {TEST_IMAGES}\n'
    f'[bold cyan]Output Dir:[/bold cyan]    {EVAL_DIR}',
    title='📂 Project Configuration'
))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[2026-07-29 15:18:43] INFO — Evaluation pipeline initialized.


INFO:PlasticSense_Eval:Evaluation pipeline initialized.


╭──────────────────────────────── 📂 Project Configuration ─────────────────────────────────╮
│ Project Root:  /content/drive/MyDrive/PlasticSense_AI                                     │
│ Model Path:    /content/drive/MyDrive/PlasticSense_AI/models/best.pt                      │
│ Test Images:   /content/drive/MyDrive/PlasticSense_AI/datasets/augmented_yolo/images/test │
│ Output Dir:    /content/drive/MyDrive/PlasticSense_AI/evaluation                          │
╰───────────────────────────────────────────────────────────────────────────────────────────╯

---
## 5. Verify Inputs (Model & Dataset)
Ensure the trained model and the test dataset exist before proceeding.

In [25]:
def verify_inputs(
    model_path: Path,
    yaml_path: Path,
    test_img_dir: Path,
    test_lbl_dir: Path
) -> Dict[str, Any]:
    """Validate all required input files and directories."""
    errors: List[str] = []

    # Verification check
    if not model_path.exists():
        errors.append(f'Model not found at: {model_path}')
    if not yaml_path.exists():
        errors.append(f'Dataset YAML not found at: {yaml_path}')
    if not test_img_dir.exists():
        errors.append(f'Test images directory missing: {test_img_dir}')
    if not test_lbl_dir.exists():
        errors.append(f'Test labels directory missing: {test_lbl_dir}')

    if errors:
        console.print('[bold red]Critical Error: Missing Required Assets[/bold red]')
        for e in errors:
            console.print(f'[red]✖ {e}[/red]')
        console.print('\n[yellow]💡 Action Required:[/yellow]')
        console.print('1. Ensure Notebook 01-06 were completed.')
        console.print('2. Check if Google Drive is mounted correctly.')
        console.print(f'3. Verify the folder structure exists in Drive: PlasticSense_AI/Datasets/augmented_yolo/')
        raise FileNotFoundError('Evaluation cannot proceed without required model and dataset files.')

    # Load dataset config
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    class_names = config.get('names', {})
    num_classes = len(class_names)

    # Count test files
    test_images = sorted(test_img_dir.glob('*.*'))
    test_labels = sorted(test_lbl_dir.glob('*.txt'))

    # Verification table
    table = Table(title='Input Verification', show_header=True)
    table.add_column('Check', style='cyan')
    table.add_column('Status', justify='right')

    table.add_row('Model Weights', f'[green]✔ {model_path.name} ({model_path.stat().st_size / 1e6:.1f} MB)[/green]')
    table.add_row('Dataset YAML', f'[green]✔ {num_classes} classes[/green]')
    table.add_row('Test Images', f'[green]✔ {len(test_images)} images[/green]')
    table.add_row('Test Labels', f'[green]✔ {len(test_labels)} label files[/green]')
    table.add_row('Classes', ', '.join(class_names.values()))

    console.print(table)
    logger.info(f'Verified: {len(test_images)} test images, {num_classes} classes.')

    return {
        'config': config,
        'class_names': class_names,
        'num_classes': num_classes,
        'test_images': test_images,
        'test_labels': test_labels
    }

# Fallback logic if weights are in the default YOLO training subfolder
if not BEST_PT_PATH.exists():
    fallback = PROJECT_ROOT / 'runs' / 'detect' / 'train' / 'weights' / 'best.pt'
    if fallback.exists():
        BEST_PT_PATH.parent.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy2(fallback, BEST_PT_PATH)
        console.print(f'[yellow]⚠ Found weights in fallback location, copied to {BEST_PT_PATH}[/yellow]')

input_info = verify_inputs(BEST_PT_PATH, DATASET_YAML, TEST_IMAGES, TEST_LABELS)
CLASS_NAMES: Dict[int, str] = input_info['class_names']
NUM_CLASSES: int = input_info['num_classes']

                                                Input Verification                                                 
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Check         ┃                                                                                          Status ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Model Weights │                                                                             ✔ best.pt (19.2 MB) │
│ Dataset YAML  │                                                                                     ✔ 8 classes │
│ Test Images   │                                                                                    ✔ 114 images │
│ Test Labels   │                                                                               ✔ 114 label files │
│ Classes       │                   plastic_bottle, plastic_bag, wrapper, styrofoam, plastic_cap, food_container, │
│               │                                                             multilayer_packaging, other_plastic │
└───────────────┴─────────────────────────────────────────────────────────────────────────────────────────────────┘

[2026-07-29 15:18:49] INFO — Verified: 114 test images, 8 classes.


INFO:PlasticSense_Eval:Verified: 114 test images, 8 classes.


---
## 6. Load Trained Model
Load the `best.pt` checkpoint — no retraining is performed.

In [26]:
def load_model(model_path: Path) -> YOLO:
    """Load a trained YOLO model from checkpoint."""
    console.print(f'[cyan]Loading model from {model_path}...[/cyan]')
    model = YOLO(str(model_path))
    model.info()
    logger.info(f'Model loaded: {model_path}')
    console.print('[bold green]✔ Model loaded successfully.[/bold green]')
    return model

model = load_model(BEST_PT_PATH)

Loading model from /content/drive/MyDrive/PlasticSense_AI/models/best.pt...

YOLO11s summary: 182 layers, 9,430,888 parameters, 0 gradients, 21.6 GFLOPs
[2026-07-29 15:19:00] INFO — Model loaded: /content/drive/MyDrive/PlasticSense_AI/models/best.pt


INFO:PlasticSense_Eval:Model loaded: /content/drive/MyDrive/PlasticSense_AI/models/best.pt


✔ Model loaded successfully.

---
## 7. Run Evaluation on Test Dataset
Execute the Ultralytics `model.val()` on the **test split** to compute all detection metrics.

In [27]:
def run_evaluation(
    model: YOLO,
    data_yaml: Path,
    device: str,
    save_dir: Path
) -> Any:
    """Run model validation on the test split."""
    console.print('[cyan]Running evaluation on TEST split...[/cyan]')
    start = time.time()

    results = model.val(
        data=str(data_yaml),
        split='test',
        device=device,
        batch=8,
        imgsz=640,
        conf=0.25,
        iou=0.5,
        plots=True,
        save_json=True,
        verbose=True
    )

    elapsed = time.time() - start
    logger.info(f'Evaluation completed in {elapsed:.1f}s')
    console.print(f'[bold green]✔ Evaluation completed in {elapsed:.1f}s[/bold green]')
    return results

eval_results = run_evaluation(model, DATASET_YAML, DEVICE, EVAL_DIR)

Running evaluation on TEST split...

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,415,896 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 154.3±209.4 MB/s, size: 1828.4 KB)
val: Scanning /content/drive/MyDrive/PlasticSense_AI/datasets/augmented_yolo/labels/test... 114 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 114/114 2.1it/s 53.5s
val: New cache created: /content/drive/MyDrive/PlasticSense_AI/datasets/augmented_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.1s/it 16.6s
                   all        114        221      0.299      0.249      0.123     0.0885
        plastic_bottle         38         63      0.444      0.381      0.226      0.145
           plastic_bag         29         41      0.333      0.293      0.137     0.0855
               wrapper         18         20      0.179       0.35     0.0791

INFO:PlasticSense_Eval:Evaluation completed in 89.5s


✔ Evaluation completed in 89.5s

---
## 8. Extract & Display Evaluation Metrics
Parse all metrics from the evaluation results including Precision, Recall, F1, mAP, TP/FP/FN counts, and per-class breakdowns.

In [28]:
def extract_metrics(results: Any, class_names: Dict[int, str]) -> Dict[str, Any]:
    """Extract comprehensive metrics from Ultralytics validation results."""
    box = results.box

    # ── Overall Metrics ──
    precision = float(box.mp)      # Mean precision
    recall    = float(box.mr)      # Mean recall
    f1_score  = 2 * (precision * recall) / (precision + recall + 1e-8)
    map50     = float(box.map50)
    map50_95  = float(box.map)

    # ── Per-Class Arrays ──
    per_class_p   = box.p.tolist() if hasattr(box.p, 'tolist') else list(box.p)
    per_class_r   = box.r.tolist() if hasattr(box.r, 'tolist') else list(box.r)
    per_class_map = box.ap50.tolist() if hasattr(box.ap50, 'tolist') else list(box.ap50)
    per_class_map_95 = box.ap.tolist() if hasattr(box.ap, 'tolist') else list(box.ap)

    # ── Per-Class F1 ──
    per_class_f1 = [
        2 * (p * r) / (p + r + 1e-8)
        for p, r in zip(per_class_p, per_class_r)
    ]

    # ── Count Test Objects per Class ──
    test_label_dir = TEST_LABELS
    class_obj_count = defaultdict(int)
    class_img_count = defaultdict(set)
    total_gt_objects = 0

    for lbl_file in test_label_dir.glob('*.txt'):
        with open(lbl_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    class_obj_count[cls_id] += 1
                    class_img_count[cls_id].add(lbl_file.stem)
                    total_gt_objects += 1

    # ── Build Per-Class Table ──
    per_class_data = []
    for i in range(len(class_names)):
        name = class_names.get(i, f'class_{i}')
        p  = per_class_p[i]  if i < len(per_class_p) else 0.0
        r  = per_class_r[i]  if i < len(per_class_r) else 0.0
        f1 = per_class_f1[i] if i < len(per_class_f1) else 0.0
        m  = per_class_map[i] if i < len(per_class_map) else 0.0
        m95 = per_class_map_95[i] if i < len(per_class_map_95) else 0.0

        per_class_data.append({
            'Class': name,
            'Precision': round(p, 4),
            'Recall': round(r, 4),
            'F1_Score': round(f1, 4),
            'mAP@50': round(m, 4),
            'mAP@50-95': round(m95, 4),
            'Num_Images': len(class_img_count.get(i, set())),
            'Num_Objects': class_obj_count.get(i, 0)
        })

    # ── Confidence Stats ──
    avg_conf = 0.0
    total_tp = 0
    total_fp = 0
    total_fn = 0

    # Estimate TP/FP/FN from per-class metrics
    for i in range(len(class_names)):
        n_obj = class_obj_count.get(i, 0)
        r_val = per_class_r[i] if i < len(per_class_r) else 0.0
        p_val = per_class_p[i] if i < len(per_class_p) else 0.0

        tp_i = int(round(r_val * n_obj))
        fn_i = n_obj - tp_i
        fp_i = int(round(tp_i / (p_val + 1e-8))) - tp_i if p_val > 0 else 0
        fp_i = max(fp_i, 0)

        total_tp += tp_i
        total_fn += fn_i
        total_fp += fp_i

    metrics = {
        'overall': {
            'Precision': round(precision, 4),
            'Recall': round(recall, 4),
            'F1_Score': round(f1_score, 4),
            'mAP@50': round(map50, 4),
            'mAP@50-95': round(map50_95, 4),
            'True_Positives': total_tp,
            'False_Positives': total_fp,
            'False_Negatives': total_fn,
            'Total_GT_Objects': total_gt_objects
        },
        'per_class': per_class_data
    }

    return metrics

metrics = extract_metrics(eval_results, CLASS_NAMES)

# ── Display Overall Metrics ──
table = Table(title='📊 Overall Test Metrics', show_header=True, header_style='bold magenta')
table.add_column('Metric', style='cyan', min_width=20)
table.add_column('Value', justify='right', style='bold')

for k, v in metrics['overall'].items():
    table.add_row(k.replace('_', ' '), str(v))

console.print(table)
logger.info(f"Overall: P={metrics['overall']['Precision']}, R={metrics['overall']['Recall']}, "
            f"F1={metrics['overall']['F1_Score']}, mAP50={metrics['overall']['mAP@50']}")

     📊 Overall Test Metrics     
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric               ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Precision            │  0.299 │
│ Recall               │ 0.2493 │
│ F1 Score             │ 0.2719 │
│ mAP@50               │  0.123 │
│ mAP@50-95            │ 0.0885 │
│ True Positives       │     49 │
│ False Positives      │    102 │
│ False Negatives      │    172 │
│ Total GT Objects     │    221 │
└──────────────────────┴────────┘

[2026-07-29 15:21:19] INFO — Overall: P=0.299, R=0.2493, F1=0.2719, mAP50=0.123


INFO:PlasticSense_Eval:Overall: P=0.299, R=0.2493, F1=0.2719, mAP50=0.123


### 8b. Per-Class Metrics Table
Display a detailed breakdown of performance for every plastic waste category.

In [29]:
# ── Per-Class Rich Table ──
pc_table = Table(title='📋 Per-Class Metrics', show_header=True, header_style='bold green')
pc_table.add_column('Class', style='cyan', min_width=22)
pc_table.add_column('Precision', justify='right')
pc_table.add_column('Recall', justify='right')
pc_table.add_column('F1 Score', justify='right')
pc_table.add_column('mAP@50', justify='right')
pc_table.add_column('mAP@50-95', justify='right')
pc_table.add_column('Images', justify='right')
pc_table.add_column('Objects', justify='right')

for row in metrics['per_class']:
    pc_table.add_row(
        row['Class'],
        str(row['Precision']),
        str(row['Recall']),
        str(row['F1_Score']),
        str(row['mAP@50']),
        str(row['mAP@50-95']),
        str(row['Num_Images']),
        str(row['Num_Objects'])
    )

console.print(pc_table)

# Save per-class CSV
df_per_class = pd.DataFrame(metrics['per_class'])
df_per_class.to_csv(METRICS_DIR / 'per_class_metrics.csv', index=False)
console.print(f"[green]✔ Saved: per_class_metrics.csv[/green]")
logger.info('Per-class metrics computed and saved.')

                                       📋 Per-Class Metrics                                       
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ Class                  ┃ Precision ┃ Recall ┃ F1 Score ┃ mAP@50 ┃ mAP@50-95 ┃ Images ┃ Objects ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ plastic_bottle         │    0.4444 │  0.381 │   0.4103 │ 0.2257 │     0.145 │     38 │      63 │
│ plastic_bag            │    0.3333 │ 0.2927 │   0.3117 │ 0.1368 │    0.0855 │     29 │      41 │
│ wrapper                │    0.1795 │   0.35 │   0.2373 │ 0.0791 │    0.0542 │     18 │      20 │
│ styrofoam              │       0.2 │ 0.2222 │   0.2105 │ 0.1517 │    0.1365 │      7 │       9 │
│ plastic_cap            │    0.3333 │   0.25 │   0.2857 │ 0.1225 │     0.098 │      4 │       4 │
│ food_container         │    0.3333 │ 0.1364 │   0.1935 │ 0.0833 │     0.063 │     20 │      22 │
│ multilayer_packaging   │    0.2692 │ 0.1129 │   0.1591 │ 0.0621 │    0.0375 │      0 │       0 │
│ other_plastic          │       0.0 │    0.0 │      0.0 │    0.0 │       0.0 │     32 │      62 │
└────────────────────────┴───────────┴────────┴──────────┴────────┴───────────┴────────┴─────────┘

✔ Saved: per_class_metrics.csv

[2026-07-29 15:21:44] INFO — Per-class metrics computed and saved.


INFO:PlasticSense_Eval:Per-class metrics computed and saved.


---
## 9. Confusion Matrix (Normal & Normalized)
Generate publication-quality confusion matrices and save as PNG, PDF, and SVG.

In [30]:
def build_confusion_matrix(
    model: YOLO,
    test_img_dir: Path,
    test_lbl_dir: Path,
    class_names: Dict[int, str],
    num_classes: int,
    conf_thresh: float = 0.25,
    iou_thresh: float = 0.5
) -> np.ndarray:
    """Build confusion matrix from predictions vs ground truth.

    Matrix shape: (num_classes+1, num_classes+1) where the last
    row = FN (background predicted) and last col = FP (background GT).
    """
    nc = num_classes
    matrix = np.zeros((nc + 1, nc + 1), dtype=np.int64)

    image_files = sorted(test_img_dir.glob('*.*'))

    for img_path in tqdm(image_files, desc='Building Confusion Matrix'):
        # Ground truth
        lbl_path = test_lbl_dir / f'{img_path.stem}.txt'
        gt_boxes = []
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        cx, cy, w, h = map(float, parts[1:5])
                        gt_boxes.append((cls_id, cx, cy, w, h))

        # Predictions
        preds = model.predict(str(img_path), conf=conf_thresh, iou=iou_thresh, verbose=False)
        pred_boxes = []
        if len(preds) > 0 and preds[0].boxes is not None:
            for box in preds[0].boxes:
                cls_pred = int(box.cls.item())
                conf_val = float(box.conf.item())
                xyxy = box.xyxy[0].cpu().numpy()
                pred_boxes.append((cls_pred, conf_val, xyxy))

        # Read image dimensions for YOLO -> pixel conversion
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        ih, iw = img.shape[:2]

        # Convert GT YOLO (cx, cy, w, h) → pixel (x1, y1, x2, y2)
        gt_pixel = []
        for cls_id, cx, cy, w, h in gt_boxes:
            x1 = (cx - w / 2) * iw
            y1 = (cy - h / 2) * ih
            x2 = (cx + w / 2) * iw
            y2 = (cy + h / 2) * ih
            gt_pixel.append((cls_id, np.array([x1, y1, x2, y2])))

        gt_matched = [False] * len(gt_pixel)

        for cls_pred, conf_val, xyxy_pred in pred_boxes:
            best_iou = 0.0
            best_idx = -1
            for j, (cls_gt, xyxy_gt) in enumerate(gt_pixel):
                if gt_matched[j]:
                    continue
                iou_val = _compute_iou(xyxy_pred, xyxy_gt)
                if iou_val > best_iou:
                    best_iou = iou_val
                    best_idx = j

            if best_iou >= iou_thresh and best_idx >= 0:
                gt_cls = gt_pixel[best_idx][0]
                matrix[gt_cls][cls_pred] += 1
                gt_matched[best_idx] = True
            else:
                # False Positive — no GT match
                matrix[nc][cls_pred] += 1

        # Unmatched GT → False Negatives
        for j, matched in enumerate(gt_matched):
            if not matched:
                gt_cls = gt_pixel[j][0]
                matrix[gt_cls][nc] += 1

    return matrix


def _compute_iou(box1: np.ndarray, box2: np.ndarray) -> float:
    """Compute IoU between two [x1,y1,x2,y2] boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter

    return inter / (union + 1e-8)


cm = build_confusion_matrix(
    model, TEST_IMAGES, TEST_LABELS,
    CLASS_NAMES, NUM_CLASSES
)
console.print(f'[green]✔ Confusion matrix built: shape {cm.shape}[/green]')

Building Confusion Matrix:   0%|          | 0/114 [00:00<?, ?it/s]

✔ Confusion matrix built: shape (9, 9)

In [32]:
def plot_confusion_matrix(
    matrix: np.ndarray,
    class_names: Dict[int, str],
    normalize: bool,
    save_dir: Path,
    prefix: str = 'confusion_matrix'
) -> None:
    """Plot and save a confusion matrix as PNG, PDF, SVG."""
    labels = [class_names[i] for i in sorted(class_names.keys())] + ['Background']

    # Convert to float64 to support normalization
    data = matrix.astype(np.float64)
    title_suffix = ''

    # Fix: Use '.0f' for float data instead of 'd'
    fmt = '.0f'

    if normalize:
        row_sums = data.sum(axis=1, keepdims=True)
        data = np.divide(data, row_sums, where=row_sums != 0)
        title_suffix = ' (Normalized)'
        fmt = '.2f'
        prefix = prefix + '_normalized'

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        data,
        annot=True,
        fmt=fmt,
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        linewidths=0.5,
        square=True,
        ax=ax,
        cbar_kws={'shrink': 0.8}
    )
    ax.set_xlabel('Predicted Class', fontsize=13, fontweight='bold')
    ax.set_ylabel('True Class', fontsize=13, fontweight='bold')
    ax.set_title(f'PlasticSense AI — Confusion Matrix{title_suffix}',
                 fontsize=15, fontweight='bold', pad=15)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()

    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(save_dir / f'{prefix}.{ext}', dpi=300, bbox_inches='tight')

    plt.show()
    plt.close(fig)
    console.print(f'[green]✔ Saved: {prefix} (PNG, PDF, SVG)[/green]')


# Normal
plot_confusion_matrix(cm, CLASS_NAMES, normalize=False, save_dir=PLOTS_DIR)

# Normalized
plot_confusion_matrix(cm, CLASS_NAMES, normalize=True, save_dir=PLOTS_DIR)

logger.info('Confusion matrices generated and saved.')

✔ Saved: confusion_matrix (PNG, PDF, SVG)

✔ Saved: confusion_matrix_normalized (PNG, PDF, SVG)

[2026-07-29 15:23:04] INFO — Confusion matrices generated and saved.


INFO:PlasticSense_Eval:Confusion matrices generated and saved.


---
## 10. Precision, Recall, PR, and F1 Curves
Generate publication-quality metric curves across confidence thresholds.

In [33]:
def plot_metric_curves(
    model: YOLO,
    test_img_dir: Path,
    test_lbl_dir: Path,
    class_names: Dict[int, str],
    save_dir: Path
) -> None:
    """Generate Precision, Recall, PR, and F1 curves across confidence thresholds."""
    conf_thresholds = np.arange(0.05, 1.0, 0.05)
    image_files = sorted(test_img_dir.glob('*.*'))
    nc = len(class_names)

    # Collect all predictions
    all_preds = []  # (img_idx, cls_pred, conf, xyxy)
    all_gts   = []  # (img_idx, cls_gt, xyxy)

    for idx, img_path in enumerate(tqdm(image_files, desc='Collecting predictions')):
        lbl_path = test_lbl_dir / f'{img_path.stem}.txt'
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        ih, iw = img.shape[:2]

        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        cx, cy, w, h = map(float, parts[1:5])
                        x1 = (cx - w/2) * iw
                        y1 = (cy - h/2) * ih
                        x2 = (cx + w/2) * iw
                        y2 = (cy + h/2) * ih
                        all_gts.append((idx, cls_id, np.array([x1, y1, x2, y2])))

        preds = model.predict(str(img_path), conf=0.01, verbose=False)
        if len(preds) > 0 and preds[0].boxes is not None:
            for box in preds[0].boxes:
                all_preds.append((
                    idx,
                    int(box.cls.item()),
                    float(box.conf.item()),
                    box.xyxy[0].cpu().numpy()
                ))

    # Compute metrics at each threshold
    precisions, recalls, f1s = [], [], []

    for thr in conf_thresholds:
        tp, fp, fn = 0, 0, 0
        # Filter preds
        filtered_preds = [(i, c, cf, b) for i, c, cf, b in all_preds if cf >= thr]

        # Group by image
        pred_by_img = defaultdict(list)
        for i, c, cf, b in filtered_preds:
            pred_by_img[i].append((c, cf, b))

        gt_by_img = defaultdict(list)
        for i, c, b in all_gts:
            gt_by_img[i].append((c, b))

        all_img_ids = set(list(pred_by_img.keys()) + list(gt_by_img.keys()))

        for img_id in all_img_ids:
            preds_img = pred_by_img.get(img_id, [])
            gts_img   = gt_by_img.get(img_id, [])
            matched   = [False] * len(gts_img)

            preds_img.sort(key=lambda x: -x[1])  # Sort by conf desc

            for p_cls, p_conf, p_box in preds_img:
                best_iou, best_j = 0, -1
                for j, (g_cls, g_box) in enumerate(gts_img):
                    if matched[j] or g_cls != p_cls:
                        continue
                    iou = _compute_iou(p_box, g_box)
                    if iou > best_iou:
                        best_iou, best_j = iou, j

                if best_iou >= 0.5 and best_j >= 0:
                    tp += 1
                    matched[best_j] = True
                else:
                    fp += 1

            fn += sum(1 for m in matched if not m)

        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f = 2 * p * r / (p + r + 1e-8)
        precisions.append(p)
        recalls.append(r)
        f1s.append(f)

    # ── Plot Individual Curves ──
    curves = {
        'precision_curve': ('Confidence Threshold', 'Precision', precisions, '#2196F3'),
        'recall_curve':    ('Confidence Threshold', 'Recall',    recalls,    '#4CAF50'),
        'f1_curve':        ('Confidence Threshold', 'F1 Score',  f1s,        '#FF9800')
    }

    for name, (xlabel, ylabel, values, color) in curves.items():
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.plot(conf_thresholds, values, color=color, linewidth=2.5, marker='o', markersize=4)
        ax.fill_between(conf_thresholds, values, alpha=0.15, color=color)
        ax.set_xlabel(xlabel, fontsize=13, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=13, fontweight='bold')
        ax.set_title(f'PlasticSense AI — {ylabel} vs {xlabel}', fontsize=14, fontweight='bold')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1.05])
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        for ext in ['png', 'pdf', 'svg']:
            fig.savefig(save_dir / f'{name}.{ext}', dpi=300)
        plt.show()
        plt.close(fig)

    # ── PR Curve ──
    fig, ax = plt.subplots(figsize=(10, 7))
    # Sort by recall for proper curve
    sorted_pairs = sorted(zip(recalls, precisions))
    r_sorted = [p[0] for p in sorted_pairs]
    p_sorted = [p[1] for p in sorted_pairs]
    ax.plot(r_sorted, p_sorted, color='#9C27B0', linewidth=2.5, marker='o', markersize=4)
    ax.fill_between(r_sorted, p_sorted, alpha=0.15, color='#9C27B0')
    ax.set_xlabel('Recall', fontsize=13, fontweight='bold')
    ax.set_ylabel('Precision', fontsize=13, fontweight='bold')
    ax.set_title('PlasticSense AI — Precision-Recall Curve', fontsize=14, fontweight='bold')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(save_dir / f'pr_curve.{ext}', dpi=300)
    plt.show()
    plt.close(fig)

    console.print('[green]✔ All metric curves saved (PNG, PDF, SVG).[/green]')

plot_metric_curves(model, TEST_IMAGES, TEST_LABELS, CLASS_NAMES, PLOTS_DIR)
logger.info('Metric curves plotted and saved.')

✔ All metric curves saved (PNG, PDF, SVG).

[2026-07-29 15:23:34] INFO — Metric curves plotted and saved.


INFO:PlasticSense_Eval:Metric curves plotted and saved.


### 10b. Per-Class mAP Bar Chart
Visualize per-class performance to easily identify strong and weak categories.

In [34]:
def plot_per_class_bar(per_class: List[Dict], save_dir: Path) -> None:
    """Bar chart of per-class mAP@50, Precision, Recall."""
    df = pd.DataFrame(per_class)
    x = np.arange(len(df))
    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.bar(x - width, df['Precision'], width, label='Precision', color='#2196F3', edgecolor='white')
    ax.bar(x,         df['Recall'],    width, label='Recall',    color='#4CAF50', edgecolor='white')
    ax.bar(x + width, df['mAP@50'],    width, label='mAP@50',   color='#FF9800', edgecolor='white')

    ax.set_xlabel('Class', fontsize=13, fontweight='bold')
    ax.set_ylabel('Score', fontsize=13, fontweight='bold')
    ax.set_title('PlasticSense AI — Per-Class Detection Performance', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df['Class'], rotation=45, ha='right', fontsize=9)
    ax.set_ylim([0, 1.1])
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(save_dir / f'per_class_performance.{ext}', dpi=300)
    plt.show()
    plt.close(fig)
    console.print('[green]✔ Per-class bar chart saved.[/green]')

plot_per_class_bar(metrics['per_class'], PLOTS_DIR)

✔ Per-class bar chart saved.

---
## 11. Sample Predictions with GT & Prediction Overlays
Display **50 random test images** with Ground Truth (green) and Predicted (red) bounding boxes overlaid.

In [35]:
# ── Color Scheme ──
GT_COLOR   = (0, 255, 0)    # Green for Ground Truth
PRED_COLOR = (0, 0, 255)    # Red for Predictions


def draw_predictions(
    img_path: Path,
    lbl_path: Path,
    model: YOLO,
    class_names: Dict[int, str],
    conf_thresh: float = 0.25
) -> np.ndarray:
    """Draw GT and predicted boxes on a test image."""
    img = cv2.imread(str(img_path))
    if img is None:
        return np.zeros((100, 100, 3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ih, iw = img.shape[:2]

    # ── Draw Ground Truth (Green) ──
    if lbl_path.exists():
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    cx, cy, w, h = map(float, parts[1:5])
                    x1 = int((cx - w/2) * iw)
                    y1 = int((cy - h/2) * ih)
                    x2 = int((cx + w/2) * iw)
                    y2 = int((cy + h/2) * ih)

                    cv2.rectangle(img, (x1, y1), (x2, y2), GT_COLOR, 2)
                    label = f'GT: {class_names.get(cls_id, str(cls_id))}'
                    cv2.putText(img, label, (x1, max(y1 - 8, 12)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, GT_COLOR, 2)

    # ── Draw Predictions (Red) ──
    preds = model.predict(str(img_path), conf=conf_thresh, verbose=False)
    if len(preds) > 0 and preds[0].boxes is not None:
        for box in preds[0].boxes:
            cls_pred = int(box.cls.item())
            conf_val = float(box.conf.item())
            xyxy = box.xyxy[0].cpu().numpy().astype(int)

            cv2.rectangle(img, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), PRED_COLOR, 2)
            label = f'Pred: {class_names.get(cls_pred, str(cls_pred))} {conf_val:.2f}'
            cv2.putText(img, label, (xyxy[0], max(xyxy[3] + 18, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, PRED_COLOR, 2)

    return img


def visualize_predictions(
    test_img_dir: Path,
    test_lbl_dir: Path,
    model: YOLO,
    class_names: Dict[int, str],
    save_dir: Path,
    n_samples: int = 50
) -> None:
    """Visualize n random test images with GT vs prediction overlays."""
    all_images = sorted(test_img_dir.glob('*.*'))
    samples = random.sample(all_images, min(n_samples, len(all_images)))

    console.print(f'[cyan]Generating {len(samples)} annotated prediction images...[/cyan]')

    # Save individual predictions
    for img_path in tqdm(samples, desc='Drawing predictions'):
        lbl_path = test_lbl_dir / f'{img_path.stem}.txt'
        annotated = draw_predictions(img_path, lbl_path, model, class_names)
        save_path = save_dir / f'pred_{img_path.stem}.png'
        cv2.imwrite(str(save_path), cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

    # Display grid (5x5 for first 25)
    display_n = min(25, len(samples))
    cols = 5
    rows = (display_n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(25, rows * 5))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for i in range(len(axes)):
        if i < display_n:
            img_path = samples[i]
            lbl_path = test_lbl_dir / f'{img_path.stem}.txt'
            annotated = draw_predictions(img_path, lbl_path, model, class_names)
            axes[i].imshow(annotated)
            axes[i].set_title(img_path.stem, fontsize=8)
        axes[i].axis('off')

    # Legend
    gt_patch   = mpatches.Patch(color='green', label='Ground Truth')
    pred_patch = mpatches.Patch(color='red',   label='Prediction')
    fig.legend(handles=[gt_patch, pred_patch], loc='upper center',
               ncol=2, fontsize=14, frameon=True)

    plt.suptitle('PlasticSense AI — Sample Test Predictions (GT=Green, Pred=Red)',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'sample_predictions_grid.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    console.print(f'[green]✔ {len(samples)} prediction images saved to {save_dir}[/green]')


visualize_predictions(
    TEST_IMAGES, TEST_LABELS, model, CLASS_NAMES,
    PREDICTIONS_DIR, n_samples=50
)
logger.info('Sample predictions visualized and saved.')

Generating 50 annotated prediction images...

Drawing predictions:   0%|          | 0/50 [00:00<?, ?it/s]

✔ 50 prediction images saved to /content/drive/MyDrive/PlasticSense_AI/evaluation/predictions

[2026-07-29 15:25:28] INFO — Sample predictions visualized and saved.


INFO:PlasticSense_Eval:Sample predictions visualized and saved.


---
## 12. Error Analysis
Systematically detect and categorize detection errors:
- **False Positives**: Predictions with no matching GT
- **False Negatives**: GT objects with no matching prediction
- **Low Confidence**: Correct predictions but with confidence < 0.4
- **Wrong Class**: Prediction matches GT location but wrong class
- **Missed Objects**: GT objects completely undetected

In [36]:
def error_analysis(
    model: YOLO,
    test_img_dir: Path,
    test_lbl_dir: Path,
    class_names: Dict[int, str],
    save_dir: Path,
    max_examples: int = 10,
    conf_thresh: float = 0.25,
    iou_thresh: float = 0.5,
    low_conf_thresh: float = 0.4
) -> Dict[str, List[Dict]]:
    """Perform detailed error analysis on test predictions."""
    errors = {
        'false_positives':      [],
        'false_negatives':      [],
        'low_confidence':       [],
        'wrong_class':          [],
        'missed_objects':       []
    }

    image_files = sorted(test_img_dir.glob('*.*'))

    for img_path in tqdm(image_files, desc='Error Analysis'):
        lbl_path = test_lbl_dir / f'{img_path.stem}.txt'
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        ih, iw = img.shape[:2]

        # Parse GT
        gt_boxes = []
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        cx, cy, w, h = map(float, parts[1:5])
                        x1 = (cx - w/2) * iw
                        y1 = (cy - h/2) * ih
                        x2 = (cx + w/2) * iw
                        y2 = (cy + h/2) * ih
                        gt_boxes.append({'cls': cls_id, 'box': np.array([x1, y1, x2, y2])})

        # Parse Predictions
        preds = model.predict(str(img_path), conf=conf_thresh, verbose=False)
        pred_boxes = []
        if len(preds) > 0 and preds[0].boxes is not None:
            for box in preds[0].boxes:
                pred_boxes.append({
                    'cls': int(box.cls.item()),
                    'conf': float(box.conf.item()),
                    'box': box.xyxy[0].cpu().numpy()
                })

        gt_matched = [False] * len(gt_boxes)

        for pred in pred_boxes:
            best_iou, best_j = 0.0, -1
            for j, gt in enumerate(gt_boxes):
                if gt_matched[j]:
                    continue
                iou = _compute_iou(pred['box'], gt['box'])
                if iou > best_iou:
                    best_iou, best_j = iou, j

            if best_iou >= iou_thresh and best_j >= 0:
                gt_matched[best_j] = True
                gt_cls = gt_boxes[best_j]['cls']

                # Wrong Class
                if pred['cls'] != gt_cls:
                    errors['wrong_class'].append({
                        'image': img_path.name,
                        'predicted': class_names.get(pred['cls'], str(pred['cls'])),
                        'actual': class_names.get(gt_cls, str(gt_cls)),
                        'confidence': round(pred['conf'], 3),
                        'iou': round(best_iou, 3)
                    })
                # Low Confidence
                elif pred['conf'] < low_conf_thresh:
                    errors['low_confidence'].append({
                        'image': img_path.name,
                        'class': class_names.get(pred['cls'], str(pred['cls'])),
                        'confidence': round(pred['conf'], 3)
                    })
            else:
                # False Positive
                errors['false_positives'].append({
                    'image': img_path.name,
                    'predicted_class': class_names.get(pred['cls'], str(pred['cls'])),
                    'confidence': round(pred['conf'], 3)
                })

        # Unmatched GT → False Negatives / Missed
        for j, matched in enumerate(gt_matched):
            if not matched:
                gt_cls = gt_boxes[j]['cls']
                errors['false_negatives'].append({
                    'image': img_path.name,
                    'missed_class': class_names.get(gt_cls, str(gt_cls))
                })
                errors['missed_objects'].append({
                    'image': img_path.name,
                    'class': class_names.get(gt_cls, str(gt_cls))
                })

    # ── Display Error Summary ──
    table = Table(title='🔍 Error Analysis Summary', show_header=True, header_style='bold red')
    table.add_column('Error Type', style='cyan', min_width=25)
    table.add_column('Count', justify='right', style='bold')

    for err_type, err_list in errors.items():
        table.add_row(err_type.replace('_', ' ').title(), str(len(err_list)))

    console.print(table)

    # ── Display Examples ──
    for err_type, err_list in errors.items():
        if err_list:
            console.print(f'\n[bold yellow]── {err_type.replace("_", " ").title()} (Top {max_examples}) ──[/bold yellow]')
            df_err = pd.DataFrame(err_list[:max_examples])
            console.print(df_err.to_string(index=False))

    # ── Save Error Images ──
    error_imgs_dir = save_dir / 'error_examples'
    error_imgs_dir.mkdir(parents=True, exist_ok=True)

    # Save a few FP and FN example images
    fp_images = list(set(e['image'] for e in errors['false_positives']))[:max_examples]
    fn_images = list(set(e['image'] for e in errors['false_negatives']))[:max_examples]

    for img_name in fp_images + fn_images:
        src = test_img_dir / img_name
        if src.exists():
            lbl_path = test_lbl_dir / f'{src.stem}.txt'
            annotated = draw_predictions(src, lbl_path, model, class_names)
            out_path = error_imgs_dir / f'error_{img_name}'
            cv2.imwrite(str(out_path), cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

    # Save error report JSON
    error_summary = {k: len(v) for k, v in errors.items()}
    error_summary['details'] = {k: v[:50] for k, v in errors.items()}
    with open(save_dir / 'error_analysis.json', 'w') as f:
        json.dump(error_summary, f, indent=4, default=str)

    console.print(f'[green]✔ Error analysis complete. Examples saved to {error_imgs_dir}[/green]')
    return errors


errors = error_analysis(
    model, TEST_IMAGES, TEST_LABELS, CLASS_NAMES,
    REPORTS_DIR, max_examples=10
)
logger.info('Error analysis completed.')

Error Analysis:   0%|          | 0/114 [00:00<?, ?it/s]

      🔍 Error Analysis Summary      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Error Type                ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ False Positives           │    98 │
│ False Negatives           │   140 │
│ Low Confidence            │     9 │
│ Wrong Class               │    35 │
│ Missed Objects            │   140 │
└───────────────────────────┴───────┘

── False Positives (Top 10) ──

image predicted_class  confidence
batch_10_000015.jpg     plastic_bag       0.530
batch_10_000015.jpg     plastic_bag       0.527
batch_10_000015.jpg     plastic_bag       0.481
batch_10_000015.jpg       styrofoam       0.437
batch_10_000044.jpg   other_plastic       0.906
batch_10_000093.jpg  plastic_bottle       0.427
batch_10_000093.jpg         wrapper       0.256
batch_11_000032.jpg     plastic_bag       0.418
batch_11_000050.jpg       styrofoam       0.490
batch_11_000096.jpg         wrapper       0.906

── False Negatives (Top 10) ──

...

── Missed Objects (Top 10) ──

image          class
batch_10_000015.jpg plastic_bottle
batch_10_000029.jpg plastic_bottle
batch_10_000044.jpg  other_plastic
batch_10_000049.jpg        wrapper
batch_10_000093.jpg        wrapper
batch_10_000093.jpg    plastic_bag
batch_11_000005.jpg      styrofoam
batch_11_000045.jpg        wrapper
batch_11_000050.jpg    plastic_bag
batch_11_000096.jpg  other_plastic

✔ Error analysis complete. Examples saved to 
/content/drive/MyDrive/PlasticSense_AI/evaluation/reports/error_examples

[2026-07-29 15:25:53] INFO — Error analysis completed.


INFO:PlasticSense_Eval:Error analysis completed.


---
## 13. Model Performance Benchmarking
Measure inference speed, FPS, GPU/CPU utilization, model size, and parameter count.

In [37]:
def benchmark_model(
    model: YOLO,
    test_img_dir: Path,
    model_path: Path,
    device: str,
    n_warmup: int = 5,
    n_benchmark: int = 50
) -> Dict[str, Any]:
    """Benchmark inference speed, FPS, resource usage, and model stats."""
    image_files = sorted(test_img_dir.glob('*.*'))
    bench_images = image_files[:min(n_benchmark + n_warmup, len(image_files))]

    # ── Warm-Up ──
    console.print('[cyan]Warming up model...[/cyan]')
    for img_path in bench_images[:n_warmup]:
        _ = model.predict(str(img_path), verbose=False)

    # ── Benchmark ──
    inference_times = []
    cpu_usages = []
    gpu_usages = []

    console.print(f'[cyan]Benchmarking on {min(n_benchmark, len(bench_images) - n_warmup)} images...[/cyan]')

    for img_path in tqdm(bench_images[n_warmup:], desc='Benchmarking'):
        cpu_before = psutil.cpu_percent(interval=None)

        start = time.perf_counter()
        _ = model.predict(str(img_path), verbose=False)
        elapsed = (time.perf_counter() - start) * 1000  # ms

        inference_times.append(elapsed)
        cpu_usages.append(psutil.cpu_percent(interval=None))

        if torch.cuda.is_available():
            try:
                gpu_util = torch.cuda.utilization(0)
                gpu_usages.append(gpu_util)
            except Exception:
                gpu_usages.append(0)

    # ── Model Stats ──
    model_size_mb = model_path.stat().st_size / (1024 * 1024)
    total_params = sum(p.numel() for p in model.model.parameters())
    trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

    avg_time = np.mean(inference_times)
    avg_fps = 1000.0 / avg_time if avg_time > 0 else 0

    performance = {
        'Avg_Inference_Time_ms': round(avg_time, 2),
        'Min_Inference_Time_ms': round(np.min(inference_times), 2),
        'Max_Inference_Time_ms': round(np.max(inference_times), 2),
        'Std_Inference_Time_ms': round(np.std(inference_times), 2),
        'Avg_FPS': round(avg_fps, 2),
        'Avg_Detection_Time_ms': round(avg_time, 2),
        'Avg_CPU_Utilization_%': round(np.mean(cpu_usages), 2) if cpu_usages else 0.0,
        'Avg_GPU_Utilization_%': round(np.mean(gpu_usages), 2) if gpu_usages else 0.0,
        'Model_Size_MB': round(model_size_mb, 2),
        'Total_Parameters': total_params,
        'Trainable_Parameters': trainable_params,
        'Device': device,
        'Images_Benchmarked': len(inference_times)
    }

    # ── Display ──
    table = Table(title='⚡ Model Performance', show_header=True, header_style='bold blue')
    table.add_column('Metric', style='cyan', min_width=28)
    table.add_column('Value', justify='right', style='bold')

    for k, v in performance.items():
        display_val = f'{v:,}' if isinstance(v, int) else str(v)
        table.add_row(k.replace('_', ' '), display_val)

    console.print(table)
    return performance


performance = benchmark_model(
    model, TEST_IMAGES, BEST_PT_PATH, DEVICE
)
logger.info(f"Benchmark: {performance['Avg_FPS']} FPS, {performance['Avg_Inference_Time_ms']} ms/img")

Warming up model...

Benchmarking on 50 images...

Benchmarking:   0%|          | 0/50 [00:00<?, ?it/s]

            ⚡ Model Performance            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric                       ┃     Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ Avg Inference Time ms        │     98.48 │
│ Min Inference Time ms        │     33.47 │
│ Max Inference Time ms        │    200.85 │
│ Std Inference Time ms        │      40.0 │
│ Avg FPS                      │     10.15 │
│ Avg Detection Time ms        │     98.48 │
│ Avg CPU Utilization %        │     56.02 │
│ Avg GPU Utilization %        │     12.56 │
│ Model Size MB                │     18.27 │
│ Total Parameters             │ 9,415,896 │
│ Trainable Parameters         │         0 │
│ Device                       │    cuda:0 │
│ Images Benchmarked           │        50 │
└──────────────────────────────┴───────────┘

[2026-07-29 15:25:59] INFO — Benchmark: 10.15 FPS, 98.48 ms/img


INFO:PlasticSense_Eval:Benchmark: 10.15 FPS, 98.48 ms/img


---
## 14. Generate & Export Reports
Save all evaluation metrics, per-class analysis, and performance benchmarks to JSON and CSV.

In [38]:
def generate_reports(
    metrics: Dict[str, Any],
    performance: Dict[str, Any],
    metrics_dir: Path,
    reports_dir: Path
) -> None:
    """Generate all JSON and CSV evaluation reports."""
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # ── 1. evaluation_report.json ──
    eval_report = {
        'project': 'PlasticSense AI',
        'notebook': '07_Model_Evaluation',
        'timestamp': timestamp,
        'model': str(BEST_PT_PATH),
        'dataset': str(DATASET_DIR),
        'split': 'test',
        'overall_metrics': metrics['overall'],
        'per_class_metrics': metrics['per_class']
    }
    with open(reports_dir / 'evaluation_report.json', 'w') as f:
        json.dump(eval_report, f, indent=4, default=str)

    # ── 2. evaluation_report.csv ──
    df_eval = pd.DataFrame([metrics['overall']])
    df_eval.insert(0, 'Timestamp', timestamp)
    df_eval.to_csv(reports_dir / 'evaluation_report.csv', index=False)

    # ── 3. per_class_metrics.csv (already saved, copy to reports too) ──
    df_pc = pd.DataFrame(metrics['per_class'])
    df_pc.to_csv(reports_dir / 'per_class_metrics.csv', index=False)

    # ── 4. performance_report.json ──
    perf_report = {
        'project': 'PlasticSense AI',
        'timestamp': timestamp,
        'performance': performance
    }
    with open(reports_dir / 'performance_report.json', 'w') as f:
        json.dump(perf_report, f, indent=4, default=str)

    # ── 5. performance_report.csv ──
    df_perf = pd.DataFrame([performance])
    df_perf.insert(0, 'Timestamp', timestamp)
    df_perf.to_csv(reports_dir / 'performance_report.csv', index=False)

    console.print('[bold green]✔ All reports generated:[/bold green]')
    report_files = [
        'evaluation_report.json',
        'evaluation_report.csv',
        'per_class_metrics.csv',
        'performance_report.json',
        'performance_report.csv'
    ]
    for rf in report_files:
        console.print(f'   📄 {rf}')


generate_reports(metrics, performance, METRICS_DIR, REPORTS_DIR)
logger.info('All reports exported.')

✔ All reports generated:

📄 evaluation_report.json

📄 evaluation_report.csv

📄 per_class_metrics.csv

📄 performance_report.json

📄 performance_report.csv

[2026-07-29 15:25:59] INFO — All reports exported.


INFO:PlasticSense_Eval:All reports exported.


---
## 15. Copy Ultralytics Auto-Generated Plots
Copy any additional plots generated by `model.val()` into our organized `plots/` directory.

In [39]:
import shutil

def copy_ultralytics_plots(save_dir: Path) -> None:
    """Copy auto-generated Ultralytics validation plots to our plots dir."""
    # Ultralytics saves val results in runs/detect/val*
    runs_dir = Path('runs/detect')
    if not runs_dir.exists():
        # Check alternative locations
        for candidate in [Path('/content/runs/detect'), PROJECT_ROOT / 'runs' / 'detect']:
            if candidate.exists():
                runs_dir = candidate
                break

    if runs_dir.exists():
        val_dirs = sorted(runs_dir.glob('val*'), key=lambda x: x.stat().st_mtime, reverse=True)
        if val_dirs:
            latest_val = val_dirs[0]
            console.print(f'[cyan]Copying plots from {latest_val}...[/cyan]')

            for plot_file in latest_val.glob('*.png'):
                dest = save_dir / f'ultralytics_{plot_file.name}'
                shutil.copy2(plot_file, dest)
                console.print(f'   📊 {plot_file.name}')

            console.print('[green]✔ Ultralytics plots copied.[/green]')
        else:
            console.print('[yellow]⚠ No val directories found in runs/detect.[/yellow]')
    else:
        console.print('[yellow]⚠ runs/detect directory not found. Skipping auto-plot copy.[/yellow]')


copy_ultralytics_plots(PLOTS_DIR)

Copying plots from runs/detect/val...

📊 BoxPR_curve.png

📊 BoxR_curve.png

📊 confusion_matrix_normalized.png

📊 BoxF1_curve.png

📊 BoxP_curve.png

📊 confusion_matrix.png

✔ Ultralytics plots copied.

---
## 16. Evaluation Output Verification
Verify that all expected output files have been generated.

In [40]:
def verify_outputs(eval_dir: Path) -> None:
    """List and verify all generated evaluation outputs."""
    table = Table(title='📦 Evaluation Outputs', show_header=True, header_style='bold cyan')
    table.add_column('Directory', style='cyan', min_width=20)
    table.add_column('File', min_width=40)
    table.add_column('Size', justify='right')

    total_files = 0
    for subdir in sorted(eval_dir.rglob('*')):
        if subdir.is_file():
            rel_dir = subdir.parent.relative_to(eval_dir)
            size_kb = subdir.stat().st_size / 1024
            size_str = f'{size_kb:.1f} KB' if size_kb < 1024 else f'{size_kb/1024:.1f} MB'
            table.add_row(str(rel_dir), subdir.name, size_str)
            total_files += 1

    console.print(table)
    console.print(f'\n[bold]Total files generated: {total_files}[/bold]')


verify_outputs(EVAL_DIR)

                               📦 Evaluation Outputs                               
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Directory              ┃ File                                        ┃     Size ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ logs                   │ evaluation.log                              │   0.8 KB │
│ metrics                │ per_class_metrics.csv                       │   0.5 KB │
│ plots                  │ confusion_matrix.pdf                        │  22.4 KB │
│ plots                  │ confusion_matrix.png                        │ 307.0 KB │
│ plots                  │ confusion_matrix.svg                        │  90.4 KB │
│ plots                  │ confusion_matrix_normalized.pdf             │  24.4 KB │
│ plots                  │ confusion_matrix_normalized.png             │ 399.6 KB │
│ plots                  │ confusion_matrix_normalized.svg             │ 112.0 KB │
│ plots                  │ f1_curve.pdf                                │  13.4 KB │
│ plots                  │ f1_curve.png                                │ 123.4 KB │
│ plots                  │ f1_curve.svg                                │  31.3 KB │
│ plots                  │ per_class_performance.pdf                   │  18.0 KB │
│ plots                  │ per_class_performance.png                   │ 192.2 KB │
│ plots                  │ per_class_performance.svg                   │  47.1 KB │
│ plots                  │ pr_curve.pdf                                │  13.0 KB │
│ plots                  │ pr_curve.png                                │ 118.8 KB │
│ plots                  │ pr_curve.svg                                │  29.0 KB │
│ plots                  │ precision_curve.pdf                         │  13.2 KB │
│ plots                  │ precision_curve.png                         │ 141.0 KB │
│ plots                  │ precision_curve.svg                         │  31.2 KB │
│ plots                  │ recall_curve.pdf                            │  13.5 KB │
│ plots                  │ recall_curve.png                            │ 134.9 KB │
│ plots                  │ recall_curve.svg                            │  31.4 KB │
│ plots                  │ sample_predictions_grid.png                 │  69.0 MB │
│ plots                  │ ultralytics_BoxF1_curve.png                 │ 181.7 KB │
│ plots                  │ ultralytics_BoxPR_curve.png                 │ 156.0 KB │
│ plots                  │ ultralytics_BoxP_curve.png                  │ 281.8 KB │
│ plots                  │ ultralytics_BoxR_curve.png                  │ 157.8 KB │
│ plots                  │ ultralytics_confusion_matrix.png            │ 221.6 KB │
│ plots                  │ ultralytics_confusion_matrix_normalized.png │ 282.1 KB │
│ predictions            │ pred_batch_10_000009.png                    │  17.3 MB │
│ predictions            │ pred_batch_10_000044.png                    │  16.0 MB │
│ predictions            │ pred_batch_10_000049.png                    │  12.8 MB │
│ predictions            │ pred_batch_10_000058.png                    │  13.4 MB │
│ predictions            │ pred_batch_11_000005.png                    │   8.3 MB │
│ predictions            │ pred_batch_11_000032.png                    │  19.2 MB │
│ predictions            │ pred_batch_11_000045.png                    │  25.4 MB │
│ predictions            │ pred_batch_11_000050.png                    │  24.4 MB │
│ predictions            │ pred_batch_12_000013.png                    │  27.5 MB │
│ predictions            │ pred_batch_12_000041.png                    │  15.8 MB │
│ predictions            │ pred_batch_12_000083.png                    │  22.5 MB │
│ predictions            │ pred_batch_12_000095.png                    │  18.0 MB │
│ predictions            │ pred_batch_13_000028.png                    │  23.1 MB │
│ predictions            │ pred_batch_13_000043.png  

Total files generated: 98

---
## 17. ✅ Final Evaluation Summary
Display a comprehensive summary of the complete evaluation pipeline.

In [41]:
def display_final_summary(
    metrics: Dict[str, Any],
    performance: Dict[str, Any],
    per_class: List[Dict]
) -> None:
    """Display the final evaluation summary panel."""
    overall = metrics['overall']

    # Best and worst class by F1
    if per_class:
        best_class = max(per_class, key=lambda x: x['F1_Score'])
        worst_class = min(per_class, key=lambda x: x['F1_Score'])
    else:
        best_class = {'Class': 'N/A', 'F1_Score': 0}
        worst_class = {'Class': 'N/A', 'F1_Score': 0}

    summary_text = (
        f"[bold green]✔ Evaluation Completed Successfully[/bold green]\n\n"
        f"[bold cyan]─── Detection Metrics ───[/bold cyan]\n"
        f"  ✔ Precision:          {overall['Precision']}\n"
        f"  ✔ Recall:             {overall['Recall']}\n"
        f"  ✔ F1 Score:           {overall['F1_Score']}\n"
        f"  ✔ mAP@50:            {overall['mAP@50']}\n"
        f"  ✔ mAP@50-95:         {overall['mAP@50-95']}\n\n"
        f"[bold cyan]─── Performance ───[/bold cyan]\n"
        f"  ✔ Avg Inference Time: {performance['Avg_Inference_Time_ms']} ms\n"
        f"  ✔ FPS:               {performance['Avg_FPS']}\n"
        f"  ✔ Model Size:        {performance['Model_Size_MB']} MB\n"
        f"  ✔ Parameters:        {performance['Total_Parameters']:,}\n\n"
        f"[bold cyan]─── Class Analysis ───[/bold cyan]\n"
        f"  ✔ Best Performing:   {best_class['Class']} (F1={best_class['F1_Score']})\n"
        f"  ✔ Worst Performing:  {worst_class['Class']} (F1={worst_class['F1_Score']})\n\n"
        f"[bold green]✔ Model Ready for Deployment[/bold green]\n\n"
        f"[bold red]Next Notebook:[/bold red] 08_Inference_and_Deployment.ipynb\n"
        f"The evaluated model will now be used for real-world\n"
        f"plastic detection on custom images."
    )

    console.print(Panel.fit(
        summary_text,
        title='🏁 PlasticSense AI — Evaluation Complete',
        border_style='bold green'
    ))


display_final_summary(metrics, performance, metrics['per_class'])
logger.info('=== EVALUATION PIPELINE COMPLETE ===')

╭───── 🏁 PlasticSense AI — Evaluation Complete ──────╮
│ ✔ Evaluation Completed Successfully                 │
│                                                     │
│ ─── Detection Metrics ───                           │
│   ✔ Precision:          0.299                       │
│   ✔ Recall:             0.2493                      │
│   ✔ F1 Score:           0.2719                      │
│   ✔ mAP@50:            0.123                        │
│   ✔ mAP@50-95:         0.0885                       │
│                                                     │
│ ─── Performance ───                                 │
│   ✔ Avg Inference Time: 98.48 ms                    │
│   ✔ FPS:               10.15                        │
│   ✔ Model Size:        18.27 MB                     │
│   ✔ Parameters:        9,415,896                    │
│                                                     │
│ ─── Class Analysis ───                              │
│   ✔ Best Performing:   plastic_bottle (F1=0.4103)   │
│   ✔ Worst Performing:  other_plastic (F1=0.0)       │
│                                                     │
│ ✔ Model Ready for Deployment                        │
│                                                     │
│ Next Notebook: 08_Inference_and_Deployment.ipynb    │
│ The evaluated model will now be used for real-world │
│ plastic detection on custom images.                 │
╰─────────────────────────────────────────────────────╯

[2026-07-29 15:25:59] INFO — === EVALUATION PIPELINE COMPLETE ===


INFO:PlasticSense_Eval:=== EVALUATION PIPELINE COMPLETE ===
